# SWISS-MODEL — Protein Structure Homology Modelling Repository

**SWISS-MODEL** is a fully automated protein structure homology-modelling server developed at the SIB Swiss Institute of Bioinformatics and the Biozentrum, University of Basel. The **SWISS-MODEL Repository** provides pre-computed comparative protein structure models for UniProtKB sequences, regularly updated with improved templates and methods.

| Property | Value |
|---|---|
| URL | https://swissmodel.expasy.org |
| Repository size | ~1M+ models |
| Quality metrics | GMQE (Global Model Quality Estimate), QMEAN (Qualitative Model Energy ANalysis) |
| Template source | Protein Data Bank (PDB) |
| Coverage | All UniProtKB/Swiss-Prot entries |

In [ ]:
import json
import time
from pathlib import Path

import requests
import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to SWISS-MODEL Repository API and confirm access
    * [x] Fetch model metadata for a set of well-studied human proteins via UniProt accession endpoint
    * [x] Retrieve quality scores (GMQE, QMEAN, coverage) for each model
    * [x] Parse into a Polars DataFrame with correct dtypes
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise model count and quality score distributions (GMQE, QMEAN)
    * [ ] Check template coverage per target protein
    * [ ] Identify proteins with high-confidence models (GMQE > 0.7)
    * [ ] Parse sequence ranges and compute coverage fractions
* [ ] **Analysis**
    * [ ] Correlate GMQE with sequence identity to template
    * [ ] Identify most-used PDB templates
    * [ ] Compare model quality across protein families
* [ ] **Visualization**
    * [ ] Scatter plot of GMQE vs. sequence identity
    * [ ] Histogram of QMEAN scores
    * [ ] Bar chart of top template organisms
* [ ] **Statistical analysis**
    * [ ] Test correlation between sequence identity and model quality
    * [ ] Compare GMQE distributions across functional categories

## 1. Ingest Data

### 1.1 Connect to SWISS-MODEL Repository API

In [ ]:
SWISSMODEL_BASE = "https://swissmodel.expasy.org/repository/uniprot"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def swissmodel_get(uniprot_id: str) -> dict:
    """Fetch SWISS-MODEL repository entry for a UniProt accession.

    Parameters
    ----------
    uniprot_id : str
        UniProt accession (e.g. "P04637" for TP53).

    Returns
    -------
    dict
        Parsed JSON response from the SWISS-MODEL repository API.
    """
    url = f"{SWISSMODEL_BASE}/{uniprot_id}.json"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    time.sleep(0.5)  # polite rate-limiting
    return r.json()


# Connectivity check: fetch the TP53 entry (P04637)
result = swissmodel_get("P04637")
print(f"UniProt: {result['result']['uniprot_id']}")
print(f"Number of models: {len(result['result']['structures'])}")
first = result["result"]["structures"][0]
print(f"\nFirst model fields: {list(first.keys())}")
print(f"  Template  : {first.get('template')}")
print(f"  GMQE      : {first.get('gmqe')}")
print(f"  QMEAN     : {first.get('qmean')}")
print(f"  Coverage  : {first.get('coverage')}")

### 1.2 Fetch Models for a Panel of Human Proteins

In [ ]:
# 20 well-studied human proteins spanning diverse functional categories
HUMAN_PROTEINS = {
    "P04637": "TP53",    # Tumour suppressor p53
    "P38398": "BRCA1",   # Breast cancer type 1 susceptibility
    "P00533": "EGFR",    # Epidermal growth factor receptor
    "P60709": "ACTB",    # Actin beta
    "P68871": "HBB",     # Haemoglobin subunit beta
    "P01308": "INS",     # Insulin
    "P01375": "TNF",     # Tumour necrosis factor
    "P05067": "APP",     # Amyloid precursor protein (Alzheimer's)
    "P10636": "MAPT",    # Microtubule-associated protein tau
    "P06239": "LCK",     # Lymphocyte-specific protein tyrosine kinase
    "P00441": "SOD1",    # Superoxide dismutase (ALS-linked)
    "P04049": "RAF1",    # RAF proto-oncogene serine/threonine-protein kinase
    "P15056": "BRAF",    # B-Raf proto-oncogene
    "P42336": "PIK3CA",  # PI3-kinase catalytic subunit alpha
    "P45985": "MAP2K4",  # Mitogen-activated protein kinase kinase 4
    "P00751": "CFB",     # Complement factor B
    "P07550": "ADRB2",   # Beta-2 adrenergic receptor
    "P35222": "CTNNB1",  # Catenin beta-1 (Wnt pathway)
    "P11362": "FGFR1",   # Fibroblast growth factor receptor 1
    "P06400": "RB1",     # Retinoblastoma-associated protein
}


def fetch_with_cache(uniprot_id: str) -> dict:
    """Return cached JSON if available, otherwise fetch from API and cache."""
    cache_path = DATA_DIR / f"swissmodel_{uniprot_id}.json"
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)
    data = swissmodel_get(uniprot_id)
    with open(cache_path, "w") as f:
        json.dump(data, f)
    return data


# Fetch all entries (uses local cache after first run)
all_results: dict[str, dict] = {}
for uid, gene in HUMAN_PROTEINS.items():
    data = fetch_with_cache(uid)
    n_models = len(data["result"]["structures"])
    all_results[uid] = data
    print(f"  {uid}  {gene:<10}  {n_models:>3} models")

print(f"\nFetched {len(all_results)} proteins.")

### 1.3 Parse into Polars DataFrame

In [ ]:
rows = []

for uid, data in all_results.items():
    res = data["result"]
    gene = HUMAN_PROTEINS[uid]
    for s in res["structures"]:
        rows.append(
            {
                "uniprot_id":    uid,
                "gene":          gene,
                "chain":         s.get("chain"),
                "template":      s.get("template"),        # PDB ID + chain, e.g. "3q05.A"
                "seq_from":      s.get("from"),            # sequence range start (1-based)
                "seq_to":        s.get("to"),              # sequence range end
                "coverage":      s.get("coverage"),        # fraction of sequence covered (0–1)
                "identity":      s.get("identity"),        # sequence identity to template (0–1)
                "similarity":    s.get("similarity"),      # sequence similarity to template (0–1)
                "gmqe":          s.get("gmqe"),            # Global Model Quality Estimate (0–1)
                "qmean":         s.get("qmean"),           # QMEAN score (typically –4 to 0)
                "created_date":  s.get("created_date"),    # ISO date string
            }
        )

df = pl.DataFrame(rows).with_columns(
    pl.col("seq_from").cast(pl.Int32),
    pl.col("seq_to").cast(pl.Int32),
    pl.col("coverage").cast(pl.Float32),
    pl.col("identity").cast(pl.Float32),
    pl.col("similarity").cast(pl.Float32),
    pl.col("gmqe").cast(pl.Float32),
    pl.col("qmean").cast(pl.Float32),
    pl.col("created_date").str.to_date(format="%Y-%m-%d", strict=False),
)

# Cache the parsed DataFrame
df.write_parquet(DATA_DIR / "swissmodel_models.parquet")

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head(5)

### 1.4 Quality Score Summary

In [ ]:
print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print()

# Null rates per column
null_counts = df.null_count().unpivot(variable_name="column", value_name="nulls")
null_rates = null_counts.with_columns(
    (pl.col("nulls") / df.shape[0] * 100).round(1).alias("null_%")
)
print("Null rates:")
print(null_rates.filter(pl.col("nulls") > 0))
print()

# Descriptive statistics for numeric quality / coverage columns
print("Quality score statistics:")
print(
    df.select(pl.col("gmqe", "qmean", "coverage", "identity")).describe()
)